# Capstone &mdash; Payment-Exception Investigation

**Day 3 &middot; the whole course, behind one endpoint**

You are building a service that investigates a payment exception and says what should happen
to it. It is accepted on a **scorecard**, not on a demo: 45 cases, an approval gate that is
checked at 100%, and cost and latency ceilings agreed in advance.

This notebook is the console. It does three things:

1. shows you the eval set, including the seven cases that separate a real agent from a lookup;
2. scores **any function you write here**, so you can iterate before you deploy anything;
3. runs the same gate against your **deployed** service.

The scoring code is `acceptance.py` and it is the same in all three places &mdash; one
implementation, three front doors. Read it: it is 250 lines of stdlib and it is the thing
your work is judged by, so it should not be a black box.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import json, os, sys, time
sys.path.insert(0, os.path.abspath("."))
sys.path.insert(0, os.path.abspath("starter"))

import acceptance
from domain import LEDGER, POLICY, NEEDS_HUMAN, SANCTIONS_WATCH, expected_recommendation

CASES = acceptance.load_cases()
APP_HOST = os.environ.get("APP_HOST", "")

print(f"{len(CASES)} cases, {len(LEDGER)} payments, {len(POLICY)} policy documents")
print("your deployed service:", f"https://{APP_HOST}" if APP_HOST else "(APP_HOST not set)")
print("ceilings:",
      f"accuracy >= {acceptance.ACCURACY_FLOOR:.0%},",
      f"grounding >= {acceptance.CITATION_FLOOR:.0%},",
      f"cost <= ${acceptance.COST_CEILING:.4f}/case,",
      f"p95 <= {acceptance.LATENCY_CEILING_P95:.0f}s")

## 1 &mdash; The eval set, and the seven cases that matter

Four outcomes: `no action`, `proceed`, `hold for a human`, `unknown`. The rule is Lab 5.5's,
unchanged. What changed is the size: Lab 7.1 measured what a seven-case eval set can
establish, which is close to nothing, so this one has 45.

Look at the last block below before you write any code.

In [ ]:
from collections import Counter
print("outcomes:", dict(Counter(c["expected"] for c in CASES)))
print()

# The cases that turn on a rule the reason code does not contain: a watchlisted
# counterparty with a completely routine failure.
trap = [c for c in CASES
        if c["watchlisted"] and c["expected"] == "hold for a human"
        and c["expected_citation"] not in NEEDS_HUMAN and c["expected_citation"]]
print(f"{len(trap)} cases are held because of WHO the counterparty is, not why it failed:")
for c in trap:
    r = LEDGER[c["ref"]]
    print(f"  {c['ref']}  {r['counterparty']:10} {r['status']:8} {r['reason_code']:20} "
          f"-> {c['expected']}")
print()
print("An agent that reads only the reason code gets 38/45 = 84.4% -- and fails the")
print("approval gate on all seven of these, which is not a score you can argue with.")

## 2 &mdash; Iterate here, before you deploy

Write a function that takes a reference and a question and returns the response contract.
`acceptance.run_local` scores it with exactly the criteria the deployed run uses, so the
number you see here is the number you will get.

Start with the deliberately terrible one below and watch it fail. Then make it less terrible.

In [ ]:
def investigate(ref: str, question: str) -> dict:
    """A baseline that answers everything the same way. It is contract-valid and useless.

    Replace this with your agents. The contract is:
      ref, recommendation, reason, citations, requires_approval, actions_taken,
      usage {cost_usd, ...}, trajectory
    """
    return {"ref": ref, "recommendation": "unknown", "reason": "not implemented",
            "citations": [], "requires_approval": False, "actions_taken": [],
            "usage": {"input_tokens": 0, "output_tokens": 0, "cost_usd": 0.0},
            "trajectory": ["baseline"]}


report, rows = acceptance.run_local(investigate)
print(acceptance.render(report, "the function above", 4))

### The obvious next step, and why it is not enough

The rule is in `domain.py`. You could call `expected_recommendation` directly, score 100%,
and learn nothing &mdash; the eval set is not a secret and the point was never to guess it.

The interesting version is the one where the **agents** derive the answer from the ledger,
the screening result and the policy documents, and the rule exists only to say who was right.
Try the reason-code-only agent below: it is what an agent that never screens the counterparty
actually produces.

In [ ]:
def reason_code_only(ref: str, question: str) -> dict:
    """Reads the reason code. Never screens the counterparty. This is the failure mode."""
    rec = LEDGER.get(ref)
    if rec is None:
        answer, cites = "unknown", []
    elif rec["status"] == "settled":
        answer, cites = "no action", []
    elif rec["reason_code"] in NEEDS_HUMAN:
        answer, cites = "hold for a human", [rec["reason_code"]]
    else:
        answer, cites = "proceed", [rec["reason_code"]]
    return {"ref": ref, "recommendation": answer, "reason": "from the reason code",
            "citations": cites, "requires_approval": answer == "hold for a human",
            "actions_taken": [],
            "usage": {"input_tokens": 0, "output_tokens": 0, "cost_usd": 0.0},
            "trajectory": ["lookup"]}


report, rows = acceptance.run_local(reason_code_only)
print(acceptance.render(report, "reason-code-only", 4))

### Read it

84.4% accuracy, and **seven approval-gate failures**. Notice which criterion is the one you
cannot argue with. Accuracy is a rate and you can always tell a story about the cases you
missed; the gate is a control, and seven payments that needed a person did not get one.

That is Module 8's distinction, arriving as a number on your own work.

## 3 &mdash; Score the deployed service

Everything above ran in this kernel. The gate that counts runs against your service on its
own hostname, through the ingress, with the probes checked first &mdash; because a service
that cannot say it is ready is not deployed.

Deploy with `../module-9/app-deploy-example.yaml`, then run this.

In [ ]:
def score_deployed(workers: int = 2, limit: int | None = None):
    """The real gate. Returns the report, and prints the scorecard."""
    if not APP_HOST:
        print("APP_HOST is not set, so there is nothing to score.")
        print("In a sandbox terminal: env | grep APP_")
        return None
    url = f"https://{APP_HOST}"
    problems = acceptance.health(url)
    if problems:
        print("health checks failed -- this is criterion zero:")
        for p in problems:
            print("  " + p)
        return None
    report, rows = acceptance.run(url, workers=workers, limit=limit)
    print(acceptance.render(report, url, workers))
    return report


# workers=2 on purpose: the gateway is shared with everyone else in the room, and the
# latency you measure at 8 concurrent callers is mostly everybody else's queue.
report = score_deployed(workers=2)

## 4 &mdash; What to fix first

Read the scorecard in this order. It is not the order the table prints in; it is the order
in which fixing something changes anything.

1. **contract** &mdash; if responses are malformed, nothing else on the card means anything.
2. **approval gate** &mdash; a control, not a rate. One failure is a failure.
3. **grounding** &mdash; if it cites the wrong document, being right was luck.
4. **accuracy** &mdash; now the number is worth reading.
5. **cost and latency** &mdash; last, because making a wrong answer cheaper is not progress.

And one thing that is not on the card: open a trace in LangFuse and read a single case
end to end. The scorecard tells you *that* something is wrong; only the trace tells you
**which hop**, which is Lab 7.4 and the difference between a fix and a guess.